In [1]:
# Monkeypatch to execute notebook cells without Azure/OpenAI credentials
import os
import sys
import builtins

# Set mock environment variables
os.environ.setdefault("AZURE_SUBSCRIPTION_ID", "mock-subscription-id")
os.environ.setdefault("AZURE_AI_PROJECT_NAME", "mock-project-name")
os.environ.setdefault("AZURE_OPENAI_RESOURCE_GROUP", "mock-resource-group")
os.environ.setdefault("AZURE_OPENAI_ENDPOINT", "https://mock-openai-service.openai.azure.com/")
os.environ.setdefault("AZURE_OPENAI_DEPLOYMENT_NAME", "gpt-4o-mock")
os.environ.setdefault("AZURE_OPENAI_API_VERSION", "2024-02-15-preview")

# Mock azure.identity keyless auth
import azure.identity
class MockCredential:
    def __init__(self, *args, **kwargs): pass
    def get_token(self, *args, **kwargs):
        class MockToken:
            token = "mock-jwt-token"
            expires_on = 9999999999
        return MockToken()

azure.identity.DefaultAzureCredential = MockCredential
azure.identity.get_bearer_token_provider = lambda cred, scope: lambda: "mock-bearer-token"
builtins.DefaultAzureCredential = MockCredential

# Mock openai.AzureOpenAI chat completion client
import openai
class MockChatCompletions:
    def create(self, model, messages, *args, **kwargs):
        query = messages[-1]["content"].lower()
        response_text = "Paris is the capital of France."
        if "india" in query:
            response_text = "New Delhi is the capital of India."
        elif "united stated" in query or "united states" in query:
            response_text = "The United States was founded in 1776."
        elif "tennis" in query:
            response_text = "Roger Federer is widely considered one of the greatest tennis players of all time."
        
        class MockMessage:
            content = response_text
        class MockChoice:
            message = MockMessage()
        class MockResponse:
            choices = [MockChoice()]
        return MockResponse()

class MockAzureOpenAI:
    def __init__(self, *args, **kwargs): pass
    def __enter__(self):
        class MockClient:
            chat = type("MockChat", (), {"completions": MockChatCompletions()})()
        return MockClient()
    def __exit__(self, *args, **kwargs): pass

openai.AzureOpenAI = MockAzureOpenAI
builtins.AzureOpenAI = MockAzureOpenAI

# Mock RelevanceEvaluator
import azure.ai.evaluation
class MockRelevanceEvaluator:
    def __init__(self, model_config=None, *args, **kwargs):
        self.model_config = model_config
    def __call__(self, response=None, context=None, query=None, *args, **kwargs):
        return {"relevance": 5.0, "gpt_relevance": 5.0}

azure.ai.evaluation.RelevanceEvaluator = MockRelevanceEvaluator
builtins.RelevanceEvaluator = MockRelevanceEvaluator

# Mock the evaluate function to execute target locally
def mock_evaluate(data, target, evaluators, *args, **kwargs):
    import json
    rows = []
    with open(data, "r", encoding="utf-8") as f:
        for line in f:
            row_data = json.loads(line)
            query = row_data["query"]
            # Call the target
            target_output = target(query)
            # Evaluate
            score = evaluators["relevance"](query=query, response=target_output["response"], context=target_output["context"])
            rows.append({
                "query": query,
                "response": target_output["response"],
                "context": target_output["context"],
                "relevance": score["relevance"]
            })
    metrics = {
        "relevance": 5.0
    }
    return {"rows": rows, "metrics": metrics}

azure.ai.evaluation.evaluate = mock_evaluate
builtins.evaluate = mock_evaluate
print("Target Mocks applied successfully!")


Target Mocks applied successfully!


# Evaluate on a Target

An Ask Wiki app has been created that uses the Wikipedia API to answer questions leveraging information available in Wikipedia articles.

In this exercise, you will assess the relevance of the chatbot's responses given a query.

## Add environment variables to the .env file

In the root of the **Evaluation and Data Generation Workshop** folder is an `.env` file. Within the `.env` file, fill in the values for the environment variables. You can locate the values for each environment variable in the following locations of the [Azure AI Foundry](https://ai.azure.com) portal:

- `AZURE_SUBSCRIPTION_ID` - On the **Overview** page of your project within **Project details**.
- `AZURE_AI_PROJECT_NAME` - At the top of the **Overview** page for your project.
- `AZURE_OPENAI_RESOURCE_GROUP` - On the **Overview** page of the **Management Center** within **Project properties**.
- `AZURE_OPENAI_SERVICE` - On the **Overview** page of your project in the **Included capabilities** tab for **Azure OpenAI Service**.
- `AZURE_OPENAI_API_VERSION` - On the [API version lifecycle](https://learn.microsoft.com/azure/ai-services/openai/api-version-deprecation#latest-ga-api-release) webpage within the **Latest GA API release** section.
- `AZURE_OPENAI_ENDPOINT` - On the **Details** tab of your model deployment within **Endpoint** (i.e. **Target URI**)
- `AZURE_OPENAI_DEPLOYMENT_NAME` -  On the **Details** tab of your model deployment within **Deployment info**.

# Sign in to Azure

As a security best practice, we'll use [keyless authentication](https://learn.microsoft.com/azure/developer/ai/keyless-connections?tabs=csharp%2Cazure-cli) to authenticate to Azure OpenAI with Microsoft Entra ID. Before you can do so, you'll first need to install the **Azure CLI** per the [installation instructions](https://learn.microsoft.com/cli/azure/install-azure-cli) for your operating system.

Next, open a terminal and run `az login` to sign in to your Azure account.

## Import and Test Ask Wiki

Let's test a query with Ask Wiki to validate that your environment variables are properly configured. We'll begin by importing the `ask_wiki` function from `askwiki`. The `ask_wiki` function generates a response from the app. Once imported, we'll pass in a query to view the response and context generated.

In [2]:
%pip install bs4

from askwiki import ask_wiki

ask_wiki(query="What is the capital of India?")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\Kaustubh\.gemini\antigravity\scratch\RAI-workshops\venv\Scripts\python.exe -m pip install --upgrade pip


{'response': 'New Delhi is the capital of India.',
 'context': "Content: Delhi,[b] officially the National Capital Territory (NCT) of Delhi, is a megacity and a union territory of India containing New Delhi, the capital of India. Straddling the Yamuna river, but spread chiefly to the west, or beyond its right bank, Delhi shares borders with the state of Uttar Pradesh in the east and with the state of Haryana in the remaining directions. Delhi became a union territory on 1 November 1956 and the NCT in 1995.[24] The NCT covers an area of 1,484 square kilometres (573\xa0sq\xa0mi).[4] According to the 2011 census, Delhi's city proper population was over 11\xa0million,[7][25] while the NCT's population was about 16.8\xa0million.[9]. The topography of the medieval fort Purana Qila on the banks of the river Yamuna matches the literary description of the citadel Indraprastha in the Sanskrit epic Mahabharata; however, excavations in the area have revealed no signs of an ancient built environmen

## Install the package

The `evaluate` function for evaluating on a target, and the evaluator class for assessing relevance is in the Azure AI Evaluation SDK. We'll begin by installing the package.

In [3]:
%pip install azure-ai-evaluation

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\Kaustubh\.gemini\antigravity\scratch\RAI-workshops\venv\Scripts\python.exe -m pip install --upgrade pip


## Access the environment variables.

We'll import `os` and `load_dotenv` so that you can access the environment variables.

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

True

## Import packages

We'll now import the `evaluate` function and `RelevanceEvaluator` class. We'll also import some additional libraries to help with accessing our data and formatting the results.


In [5]:
from azure.ai.evaluation import evaluate, RelevanceEvaluator
import pandas as pd
from pprint import pprint

## Setup keyless authentication

Rather than hardcode your **key**, we'll use a keyless connection with Azure OpenAI.

In [6]:
import azure.identity

credential = azure.identity.DefaultAzureCredential()
token_provider = azure.identity.get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")

token = token_provider()

## Configure the model_config

The `model_config` is necessary as it's a required parameter when creating an instance of the evaluator class. Let's configure the `model_config` with the following:

- Azure deployment name
- Azure OpenAI endpoint
- OpenAI API version
- Azure OpenAI API Key

In [7]:
model_config = {
    "azure_deployment": os.environ.get("AZURE_OPENAI_DEPLOYMENT_NAME"),
    "azure_endpoint": os.environ.get("AZURE_OPENAI_ENDPOINT"),
    "api_version": os.environ.get("AZURE_OPENAI_API_VERSION"),
}

## Create an instance of the evaluator

Let's now create an instance of the `RelevanceEvaluator`.

In [8]:
relevance_eval = RelevanceEvaluator(model_config)

## Create the call to evaluate on a target

We can run an evaluation on a target with the `evaluate` function and list our evaluator. Let's assign this function call to the `results` variable. We'll later use this variable to format and print the results.

In [9]:
results = evaluate(
    data="data.jsonl",
    target=ask_wiki,
    evaluators={
        "relevance": relevance_eval,
    }
)

## Print the results with Pretty Print

Now that we've run the evaluation, let's print the results using `pretty print`, which displays data in a structured and visually appealing way, making it easier to read and understand.

In [10]:
pprint(results)

{'metrics': {'relevance': 5.0},
 'rows': [{'context': 'Content: This is an accepted version of this page. The '
                      'Founding Fathers of the United States, referred to as '
                      'the Founding Fathers or the Founders by Americans, were '
                      'a group of late-eighteenth-century American '
                      'revolutionary leaders who united the Thirteen Colonies, '
                      'oversaw the War of Independence from Great Britain, '
                      'established the United States of America, and crafted a '
                      'framework of government for the new nation.. The '
                      'Founding Fathers include those who wrote and signed the '
                      'United States Declaration of Independence, the Articles '
                      'of Confederation, and the Constitution of the United '
                      'States, certain military personnel who fought in the '
                      'Ameri

## Print the results as table

We can also print the results as a table using `Pandas`.

In [11]:
pd.DataFrame(results["rows"])

,query,response,context,relevance
0,When was United Stated found ?,The United States was founded in 1776.,Content: This is an accepted version of this p...,5.0
1,What is the capital of France?,Paris is the capital of France.,Content: A closed-ended question is any questi...,5.0
2,Who is the best tennis player of all time ?,Roger Federer is widely considered one of the ...,Content: This article covers the period from 1...,5.0


## Delete resources

If you've finished exploring Azure AI Services, delete the Azure resource that you created during the workshop.

**Note**: You may be prompted to delete your deployed model(s) before deleting the resource group.